In [1]:
import os, json, time, requests
from datetime import datetime, timezone

URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS?"
headers = {"Ocp-Apim-Subscription-Key": os.environ["API_KEY"]}

for flow in ["M", "X"]:
    os.makedirs(f"data_{flow}", exist_ok=True)

    for year in range(1994, 2027):
        params = {
            "reporterCode": 818,
            "flowCode": flow,
            "period": str(year),
            "cmdCode": "AG6",
            "maxRecords": 100000,
            "includeDesc": "true"
        }
        print(f"Fetching {flow} - {year}")
        response = requests.get(URL, headers=headers, params=params)
        print("status code:", response.status_code)

        if response.status_code == 200:
            records = response.json().get("data", [])
            print("Records:", len(records))

            if records:
                extracted_at = datetime.now(timezone.utc).isoformat()
                for row in records:
                    row["_ingested_at"] = extracted_at
                    row["_flow"] = flow

                with open(f"data_{flow}/{year}.json", "w") as f:
                    json.dump(records, f)

        time.sleep(5)

Fetching M - 1994
status code: 200
Records: 31468
Fetching M - 1995
status code: 200
Records: 30227
Fetching M - 1996
status code: 200
Records: 32425
Fetching M - 1997
status code: 200
Records: 35365
Fetching M - 1998
status code: 200
Records: 38096
Fetching M - 1999
status code: 200
Records: 37438
Fetching M - 2000
status code: 200
Records: 37217
Fetching M - 2001
status code: 200
Records: 38518
Fetching M - 2002
status code: 200
Records: 37262
Fetching M - 2003
status code: 200
Records: 35827
Fetching M - 2004
status code: 200
Records: 36724
Fetching M - 2005
status code: 200
Records: 40341
Fetching M - 2006
status code: 200
Records: 37987
Fetching M - 2007
status code: 200
Records: 38693
Fetching M - 2008
status code: 200
Records: 59548
Fetching M - 2009
status code: 200
Records: 63908
Fetching M - 2010
status code: 200
Records: 62981
Fetching M - 2011
status code: 200
Records: 57200
Fetching M - 2012
status code: 200
Records: 58221
Fetching M - 2013
status code: 200
Records: 56449


In [2]:
import os, glob
from sqlalchemy import create_engine, text

engine = create_engine(f"postgresql+psycopg://postgres:{os.environ['POSTGRES_PASSWORD']}@db:5432/postgres")

with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS bronze"))
    conn.execute(text("CREATE TABLE IF NOT EXISTS bronze.comtrade_raw (file TEXT, payload JSONB)"))
    conn.execute(text("TRUNCATE bronze.comtrade_raw"))  # avoids duplicates on reruns

    for path in sorted(glob.glob("data_*/*.json")):  # reads both data_M and data_X
        with open(path) as f:
            payload = f.read()
        conn.execute(text("INSERT INTO bronze.comtrade_raw VALUES (:f, CAST(:p AS JSONB))"),
                        {"f": path, "p": payload})
        print("Loaded", path)

Loaded data_M/1994.json
Loaded data_M/1995.json
Loaded data_M/1996.json
Loaded data_M/1997.json
Loaded data_M/1998.json
Loaded data_M/1999.json
Loaded data_M/2000.json
Loaded data_M/2001.json
Loaded data_M/2002.json
Loaded data_M/2003.json
Loaded data_M/2004.json
Loaded data_M/2005.json
Loaded data_M/2006.json
Loaded data_M/2007.json
Loaded data_M/2008.json
Loaded data_M/2009.json
Loaded data_M/2010.json
Loaded data_M/2011.json
Loaded data_M/2012.json
Loaded data_M/2013.json
Loaded data_M/2014.json
Loaded data_M/2015.json
Loaded data_M/2016.json
Loaded data_M/2017.json
Loaded data_M/2018.json
Loaded data_M/2019.json
Loaded data_M/2020.json
Loaded data_M/2021.json
Loaded data_M/2022.json
Loaded data_M/2023.json
Loaded data_M/2024.json
Loaded data_M/2025.json
Loaded data_X/1994.json
Loaded data_X/1995.json
Loaded data_X/1996.json
Loaded data_X/1997.json
Loaded data_X/1999.json
Loaded data_X/2000.json
Loaded data_X/2001.json
Loaded data_X/2002.json
Loaded data_X/2003.json
Loaded data_X/20

In [5]:
import os, json, time, requests
from datetime import datetime, timezone

URL = "https://api.frankfurter.dev/v2/rates"
os.makedirs("data_fx", exist_ok=True)

for year in range(1994, 2027):
    params = {"base": "usd", "quotes": "egp",
                "from": f"{year}-01-01", "to": f"{year}-12-31"}
    print(f"Fetching FX - {year}")
    response = requests.get(URL, params=params)
    print("status code:", response.status_code)

    if response.status_code == 200:
        records = response.json()
        print("Records:", len(records))

        if records:
            extracted_at = datetime.now(timezone.utc).isoformat()
            for row in records:
                row["_ingested_at"] = extracted_at

            with open(f"data_fx/{year}.json", "w") as f:
                json.dump(records, f)

    time.sleep(10)

Fetching FX - 1994
status code: 200
Records: 283
Fetching FX - 1995
status code: 200
Records: 365
Fetching FX - 1996
status code: 200
Records: 365
Fetching FX - 1997
status code: 200
Records: 363
Fetching FX - 1998
status code: 200
Records: 355
Fetching FX - 1999
status code: 200
Records: 363
Fetching FX - 2000
status code: 200
Records: 358
Fetching FX - 2001
status code: 200
Records: 359
Fetching FX - 2002
status code: 200
Records: 358
Fetching FX - 2003
status code: 200
Records: 355
Fetching FX - 2004
status code: 200
Records: 363
Fetching FX - 2005
status code: 200
Records: 363
Fetching FX - 2006
status code: 200
Records: 308
Fetching FX - 2007
status code: 200
Records: 338
Fetching FX - 2008
status code: 200
Records: 359
Fetching FX - 2009
status code: 200
Records: 343
Fetching FX - 2010
status code: 200
Records: 272
Fetching FX - 2011
status code: 200
Records: 339
Fetching FX - 2012
status code: 200
Records: 366
Fetching FX - 2013
status code: 200
Records: 365
Fetching FX - 2014
s

In [6]:
import os, glob
from sqlalchemy import create_engine, text

engine = create_engine(f"postgresql+psycopg://postgres:{os.environ['POSTGRES_PASSWORD']}@db:5432/postgres")

with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS bronze"))
    conn.execute(text("CREATE TABLE IF NOT EXISTS bronze.fx_rates_raw (file TEXT, payload JSONB)"))
    conn.execute(text("TRUNCATE bronze.fx_rates_raw"))  # avoids duplicates on reruns

    for path in sorted(glob.glob("data_fx/*.json")):  # one file per year
        with open(path) as f:
            payload = f.read()
        conn.execute(text("INSERT INTO bronze.fx_rates_raw VALUES (:f, CAST(:p AS JSONB))"),
                        {"f": path, "p": payload})
        print("Loaded", path)

Loaded data_fx/1994.json
Loaded data_fx/1995.json
Loaded data_fx/1996.json
Loaded data_fx/1997.json
Loaded data_fx/1998.json
Loaded data_fx/1999.json
Loaded data_fx/2000.json
Loaded data_fx/2001.json
Loaded data_fx/2002.json
Loaded data_fx/2003.json
Loaded data_fx/2004.json
Loaded data_fx/2005.json
Loaded data_fx/2006.json
Loaded data_fx/2007.json
Loaded data_fx/2008.json
Loaded data_fx/2009.json
Loaded data_fx/2010.json
Loaded data_fx/2011.json
Loaded data_fx/2012.json
Loaded data_fx/2013.json
Loaded data_fx/2014.json
Loaded data_fx/2015.json
Loaded data_fx/2016.json
Loaded data_fx/2017.json
Loaded data_fx/2018.json
Loaded data_fx/2019.json
Loaded data_fx/2020.json
Loaded data_fx/2021.json
Loaded data_fx/2022.json
Loaded data_fx/2023.json
Loaded data_fx/2024.json
Loaded data_fx/2025.json
Loaded data_fx/2026.json
